# Rare SNP Analysis Pipeline
## Extraction, Mapping, and Characterization of Rare Variants

This notebook documents the complete pipeline for extracting rare single nucleotide polymorphisms (SNPs) from GnomAD v3.0, mapping them to genomic features, and analyzing their nucleotide context.

## Table of Contents

1. [Overview & Workflow](#overview)
2. [Part 1: Extract Rare SNPs from GnomAD v3.0](#part1)
3. [Part 2: Prepare SNP Data](#part2-prep)
4. [Part 3: Map Rare SNPs to Genomic Features](#part3-mapping)
   - [Option A: TSS-Aligned Forward Mapping](#option-a)
   - [Option B: TSS-Aligned Backward Mapping](#option-b)
   - [Option C: Exon-Aligned Backward Mapping](#option-c)
5. [Part 4: Count SNPs by Position](#part4-count)
6. [Part 5: Analyze Nucleotide Content & CpG Context](#part5-nucleotides)
7. [Part 6: Parse CpG vs Non-CpG Mutations](#part6-cpg)

## Overview & Workflow

### Pipeline Summary

This analysis extracts rare variants (allele frequency < 0.1% or 0.001%) from gnomAD v3.0 whole-genome sequencing data and maps them to regulatory regions of interest. The pipeline includes:

1. **Data Source**: GnomAD v3.0 (publicly available on Google Cloud, reference genome GRCh38)
2. **Variant Extraction**: SNPs with allele frequency < 0.001 (0.1%) using Hail
3. **Mapping**: Map SNPs to positions relative to transcription start sites (TSS) or exon boundaries
4. **Quantification**: Count SNPs and analyze nucleotide composition at each position
5. **Annotation**: Track CpG dinucleotide context to identify CpG-prone mutations

### Key Inputs

- **gnomAD VCF files**: `gs://gcp-public-data--gnomad/release/3.0/vcf/genomes/` (one per chromosome)
- **TSS annotation**: Fantom5 CAGE sequencing data (TSSs with strand information)
- **Reference genome**: Homo_sapiens.GRCh38.dna.toplevel.fa (Ensembl)
- **Gene annotations**: Gencode v40 (optional, for filtering by gene type)

### Software Requirements

```
- Hail (Python, for VCF processing)
- bedtools (for FASTA extraction)
- pandas (for data aggregation)
- Python 3.7+
```

## Part 1: Extract Rare SNPs from GnomAD v3.0

Load whole-genome VCF files from GnomAD, filter for SNPs with high quality and allele frequency < 0.1%, and export to TSV format.

In [ ]:
import hail as hl

# Initialize Hail with Google Cloud Storage credentials
hl.init(spark_conf={
    'spark.jars': 'file:///home/alexpalazzo1/hail/jars/gcs-connector-hadoop3-latest.jar',
    'spark.hadoop.google.cloud.auth.service.account.enable': 'true',
    'spark.hadoop.google.cloud.auth.service.account.json.keyfile': '/home/alexpalazzo1/hail/gnomad-hail-439921-1ae1364a1b5a.json',
    'spark.hadoop.fs.gs.impl': 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem',
    'spark.hadoop.fs.AbstractFileSystem.gs.impl': 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS'
})

print('Hail initialized successfully.')

### Import gnomAD VCF Files

Load VCF files for all chromosomes from GnomAD v3.0 into a single Hail MatrixTable.

In [ ]:
# List of VCF files stored in Google Cloud for all chromosomes
vcf_files = [f'gs://gcp-public-data--gnomad/release/3.0/vcf/genomes/gnomad.genomes.r3.0.sites.chr{chrom}.vcf.bgz' 
             for chrom in list(range(1, 23)) + ['X', 'Y']]

# Load all VCF files into a single MatrixTable
mt = hl.import_vcf(vcf_files, min_partitions=4, reference_genome='GRCh38')

print(f'Total variants loaded: {mt.count_rows():,}')

### Filter for SNPs and High-Quality Variants

Apply quality filters to retain only high-confidence SNPs with observed alleles.

In [ ]:
# Filter for SNPs only (biallelic, single nucleotide substitutions)
mt = mt.filter_rows(hl.is_snp(mt.alleles[0], mt.alleles[1]))
print(f'SNPs only: {mt.count_rows():,}')

# Filter high-quality variants (no quality filters applied by gnomAD)
mt_filtered = mt.filter_rows(hl.len(mt.filters) == 0)
print(f'After quality filtering (no gnomAD filters): {mt_filtered.count_rows():,}')

# Remove AC0 variants (allele count = 0, not observed)
mt_filtered = mt_filtered.filter_rows(~mt_filtered.filters.contains('AC0'))
print(f'After removing AC0 variants: {mt_filtered.count_rows():,}')

# Ensure allele frequency > 0
mt_filtered = mt_filtered.filter_rows(mt_filtered.info.AF[0] > 0.0)
print(f'With AF > 0: {mt_filtered.count_rows():,}')

### Filter for Rare Variants

Select only SNPs with allele frequency < 0.1% (AF < 0.001). You can modify this threshold as needed.

In [ ]:
# Filter for rare variants: AF < 0.1% (0.001)
# To use AF < 1% instead, change 0.001 to 0.01
AF_THRESHOLD = 0.001

mt_rare = mt_filtered.filter_rows(mt_filtered.info.AF[0] < AF_THRESHOLD)
print(f'Variants with AF < {AF_THRESHOLD*100}%: {mt_rare.count_rows():,}')

### Extract and Export Results

Select relevant fields and export to TSV format.

In [ ]:
# Annotate with allele frequency
mt_af = mt_rare.annotate_rows(af=mt_rare.info.AF[0])

# Select relevant columns
results = mt_af.select_rows(
    chromosome=mt_af.locus.contig,
    position=mt_af.locus.position,
    ref_allele=mt_af.alleles[0],
    alt_allele=mt_af.alleles[1],
    allele_frequency=mt_af.af
)

# Export to TSV
output_file = '/home/alexpalazzo1/Documents/Tina/Rare_SNP/gnomad_AF_less_than_0.1percent.tsv'
results_table = results.rows()
results_table.export(output_file)

print(f'Export complete: {output_file}')
print(f'Total rare variants exported: {results_table.count():,}')
print('\nFirst 5 variants:')
results_table.show(5)

# Cleanup
hl.stop()

---
## Part 2: Prepare SNP Data

Convert the reference and alternate alleles into a standardized mutation notation (e.g., 'A>G').

In [ ]:
import csv

def join_allele_columns(input_file, output_file):
    """
    Join reference allele (column 3) and alternate allele (column 4) 
    into mutation notation (column 3: REF>ALT).
    Remove the original alternate allele column (column 4).
    
    Args:
        input_file: TSV file with alleles in columns 3 and 4
        output_file: Output TSV file with joined mutation column
    """
    with open(input_file, 'r', newline='') as infile, open(output_file, 'w', newline='') as outfile:
        reader = csv.reader(infile, delimiter='\\t')
        writer = csv.writer(outfile, delimiter='\\t')
        
        for row in reader:
            # Join columns 3 and 4 (indices 2 and 3) with '>'
            row[2] = f'{row[2]}>{row[3]}'
            # Remove the original alternate allele column
            row.pop(3)
            # Write modified row
            writer.writerow(row)

# Example usage
input_file = 'FA_all_pop_snp_rare.tsv'
output_file = 'FA_all_pop_snp_rare_defined.tsv'
join_allele_columns(input_file, output_file)
print(f'Alleles joined. Output: {output_file}')

---
## Part 3: Map Rare SNPs to Genomic Features

Map SNPs to their relative positions within regions of interest (e.g., 1kb around TSS). **Choose ONE of the following mapping approaches based on your analysis goals.**

### Summary of Mapping Approaches

| Option | Feature | Position Mode | Use Case |
|--------|---------|---------------|----------|
| **A** | TSS | Forward (from start) | TSS-proximal analysis |
| **B** | TSS | Backward (from end) | Promoter-proximal regions |
| **C** | Exon boundary | Backward (from end) | Exon-structure analysis |

**Choose one option for your analysis.**

### Core Mapping Function (Used by All Options)

This function handles the actual mapping logic, accounting for strand orientation.

In [ ]:
import concurrent.futures
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def read_annotation_file(file_path):
    """
    Read annotation file (TSS or exon data) into memory.
    Args: file_path (str): Path to annotation file (BED/GFF format)
    Returns: dict: {chromosome: [(start, end, direction), ...]}
    """
    data = {}
    try:
        with open(file_path, 'r') as file:
            for line in file:
                parts = line.strip().split()
                chr_num = parts[0].replace('chr', '')
                start = int(parts[1])
                end = int(parts[2])
                direction = parts[5]
                if chr_num not in data:
                    data[chr_num] = []
                data[chr_num].append((start, end, direction))
    except Exception as e:
        logging.error(f'Error reading annotation file: {e}')
        raise
    return data


def process_snp_chunk(chunk, annotation_data, position_mode='forward'):
    """
    Process a batch of SNP lines and map to annotation features.
    Args:
        chunk: List of SNP lines
        annotation_data: Dictionary of features
        position_mode: 'forward' or 'backward'
    Returns: List of mapped lines
    """
    results = []
    for line in chunk:
        try:
            parts = line.strip().split()
            chr_num = parts[0].replace('chr', '')
            coord = int(parts[1])
            extra_data = parts[2:]
            
            if chr_num in annotation_data:
                for start, end, direction in annotation_data[chr_num]:
                    if direction == '+':
                        if start <= coord <= end:
                            range_position = coord - start if position_mode == 'forward' else coord - end
                            results.append(f'{range_position} {\" \".join(extra_data)} {chr_num} {coord} {direction}\\n')
                            break
                    elif direction == '-':
                        if end <= coord <= start:
                            range_position = start - coord if position_mode == 'forward' else end - coord
                            results.append(f'{range_position} {\" \".join(extra_data)} {chr_num} {coord} {direction}\\n')
                            break
        except Exception as e:
            logging.error(f'Error processing line: {line.strip()}')
    return results


def map_snps_to_features(snp_file, annotation_file, output_file, position_mode='forward', chunk_size=100000):
    """
    Map SNPs to their relative positions within genomic features.
    """
    annotation_data = read_annotation_file(annotation_file)
    logging.info(f'Loaded {sum(len(v) for v in annotation_data.values()):,} features')

    try:
        with open(snp_file, 'r') as snp_input, open(output_file, 'w') as snp_output:
            next(snp_input)
            with concurrent.futures.ProcessPoolExecutor() as executor:
                chunk = []
                futures = []
                for line in snp_input:
                    chunk.append(line)
                    if len(chunk) == chunk_size:
                        futures.append(executor.submit(process_snp_chunk, chunk, annotation_data, position_mode))
                        chunk = []
                if chunk:
                    futures.append(executor.submit(process_snp_chunk, chunk, annotation_data, position_mode))
                for future in concurrent.futures.as_completed(futures):
                    results = future.result()
                    snp_output.writelines(results)
        logging.info(f'Mapping complete: {output_file}')
    except Exception as e:
        logging.error(f'Error during mapping: {e}')
        raise

print('Mapping functions defined.')

### Option A: TSS-Aligned Forward Mapping

Map SNPs to relative distance from the **transcription start site (TSS)**. Position 0 = TSS. Use this for analyzing SNP distribution in TSS-proximal regions.

In [ ]:
# Configure for Option A
snp_file_a = 'FA_all_pop_snp_rare_defined.tsv'
annotation_file_a = 'lncRNA/results/lncrna_non_testis_tss_hg38_lv3_w500.bed'
output_file_a = 'lncRNA/results/rareSNP_mapped_lncrna_non_testis.txt'

# Uncomment to run:
# map_snps_to_features(snp_file_a, annotation_file_a, output_file_a, position_mode='forward')

### Option B: TSS-Aligned Backward Mapping

Map SNPs relative distance measured **backward from TSS**. Useful for analyzing 5' regulatory regions.

In [ ]:
# Configure for Option B
snp_file_b = 'FA_all_pop_snp_rare_defined.tsv'
annotation_file_b = 'Fantom5_CAGE_hg38_gencode.v40_TSS_EIB_Updated_TSStoEIB+.txt'
output_file_b = 'exon_aligned/rareSNP_mapped_TSStoEIB+_backward.txt'

# Uncomment to run:
# map_snps_to_features(snp_file_b, annotation_file_b, output_file_b, position_mode='backward')

### Option C: Exon-Aligned Backward Mapping

Map SNPs relative to **exon boundaries**. Useful for analyzing SNP patterns relative to transcribed regions.

In [ ]:
# Configure for Option C
snp_file_c = 'FA_all_pop_snp_rare_defined.tsv'
annotation_file_c = 'exon_aligned/Fantom5_CAGE_hg38_gencode.v40_TSS_EIB_Updated_TSStoEIB+.txt'
output_file_c = 'exon_aligned/rareSNP_mapped_TSStoEIB+_backward.txt'

# Uncomment to run:
# map_snps_to_features(snp_file_c, annotation_file_c, output_file_c, position_mode='backward')

### Optional: Reverse Complement for Forward Mapping

If using **Option A**, convert minus strand SNPs to reverse complement.

In [ ]:
reverse_complement_map = {
    'A>C': 'T>G', 'A>T': 'T>A', 'A>G': 'T>C', 
    'C>T': 'G>A', 'C>A': 'G>T', 'C>G': 'G>C',
    'G>A': 'C>T', 'G>T': 'C>A', 'G>C': 'C>G',
    'T>A': 'A>T', 'T>C': 'A>G', 'T>G': 'A>C'
}

def apply_reverse_complement(input_file, output_file, rc_map):
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        for line in infile:
            line = line.strip()
            if line.endswith('+'):
                outfile.write(line + '\\n')
            elif line.endswith('-'):
                parts = line.split()
                mutation = parts[1]
                if mutation in rc_map:
                    parts[1] = rc_map[mutation]
                outfile.write(' '.join(parts) + '\\n')

# Example (uncomment to use):
# input_file = 'lncRNA/results/rareSNP_mapped_lncrna_non_testis.txt'
# output_file = 'lncRNA/results/rareSNP_mapped_lncrna_non_testis_strand_complement.txt'
# apply_reverse_complement(input_file, output_file, reverse_complement_map)

---
## Part 4: Count SNPs by Position

Aggregate SNP counts by position and mutation type.

In [ ]:
import pandas as pd

def count_snps_by_position(input_file, output_csv, position_range=(0, 1000)):
    """
    Count SNPs by position and mutation type.
    """
    mutation_types = ['A>G', 'T>C', 'C>G', 'T>G', 'C>A', 'A>T', 
                     'G>C', 'G>T', 'C>T', 'T>A', 'A>C', 'G>A']
    
    counts = {i: {mutation: 0 for mutation in mutation_types} 
             for i in range(position_range[0], position_range[1] + 1)}
    
    with open(input_file, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if not parts or parts[0] == 'NA':
                continue
            try:
                position = int(parts[0])
                mutation = parts[1]
                if position_range[0] <= position <= position_range[1] and mutation in mutation_types:
                    counts[position][mutation] += 1
            except (ValueError, IndexError):
                continue
    
    df = pd.DataFrame.from_dict(counts, orient='index', columns=mutation_types)
    df.index.name = 'Position'
    df.to_csv(output_csv)
    
    print(f'SNP counting complete. Total SNPs: {df.sum().sum():,}')
    print(f'Output: {output_csv}')
    return df

# Example:
# input_file = 'lncRNA/results/rareSNP_mapped_lncrna_non_testis_strand_complement.txt'
# output_csv = 'lncRNA/results/rareSNP_counts_by_position.csv'
# df_counts = count_snps_by_position(input_file, output_csv)

---
## Part 5: Analyze Nucleotide Content with CpG Context

Extract nucleotide sequences and count composition at each position, tracking CpG dinucleotides.

### Step 1: Convert to GFF Format

In [ ]:
def convert_to_gff(input_file, output_file):
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        for line in infile:
            if not line.strip():
                continue
            fields = line.strip().split('\\t')
            if len(fields) >= 6:
                chromosome = fields[0].replace('chr', '')
                start = fields[1]
                end = fields[2]
                strand = fields[5]
                outfile.write(f'{chromosome}\\t{start}\\t{end}\\t.\\t.\\t{strand}\\n')
    print(f'GFF conversion complete: {output_file}')

# Example:
# convert_to_gff('lncrna.bed', 'lncrna.gff')

### Step 2: Extract FASTA Sequences

Use bedtools to extract sequences from the reference genome:
```bash
bedtools getfasta -fi reference.fa -bed regions.gff -fo output.fa -s -name
```

### Step 3: Count Nucleotides with CpG Tracking

In [ ]:
import os

def calculate_nucleotide_counts_with_cpg(sequences_with_strands):
    if not sequences_with_strands:
        return {}
    sequence_length = len(sequences_with_strands[0][0])
    counts = {
        'A': [0] * sequence_length,
        'T': [0] * sequence_length,
        'G': [0] * sequence_length,
        'C': [0] * sequence_length,
        'C_in_CpG': [0] * sequence_length,
        'G_in_CpG': [0] * sequence_length,
        'C_not_in_CpG': [0] * sequence_length,
        'G_not_in_CpG': [0] * sequence_length
    }
    for sequence, strand in sequences_with_strands:
        for i, base in enumerate(sequence):
            target_index = i - 1 if strand == '-' else i
            if 0 <= target_index < sequence_length:
                upper_base = base.upper()
                if upper_base == 'A':
                    counts['A'][target_index] += 1
                elif upper_base == 'T':
                    counts['T'][target_index] += 1
                elif upper_base == 'G':
                    counts['G'][target_index] += 1
                    if i > 0 and sequence[i-1].upper() == 'C':
                        counts['G_in_CpG'][target_index] += 1
                    else:
                        counts['G_not_in_CpG'][target_index] += 1
                elif upper_base == 'C':
                    counts['C'][target_index] += 1
                    if i < len(sequence) - 1 and sequence[i+1].upper() == 'G':
                        counts['C_in_CpG'][target_index] += 1
                    else:
                        counts['C_not_in_CpG'][target_index] += 1
    return counts

def read_fasta_with_strand(fasta_file):
    sequences = []
    if not os.path.exists(fasta_file):
        logging.error(f'File not found: {fasta_file}')
        return []
    with open(fasta_file, 'r') as f:
        seq = ''
        strand = '+'
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    sequences.append((seq, strand))
                seq = ''
                strand = '-' if '(-)' in line.lower() or 'reverse' in line.lower() else '+'
            else:
                seq += line
        if seq:
            sequences.append((seq, strand))
    return sequences

def save_nucleotide_counts(counts, output_file):
    if not counts:
        return
    with open(output_file, 'w') as f:
        f.write('Position\\tA\\tT\\tC\\tG\\tC_in_CpG\\tG_in_CpG\\tC_not_in_CpG\\tG_not_in_CpG\\n')
        for i in range(len(counts['A'])):
            f.write(f'{i + 1}\\t{counts["A"][i]}\\t{counts["T"][i]}\\t{counts["C"][i]}\\t{counts["G"][i]}\\t{counts["C_in_CpG"][i]}\\t{counts["G_in_CpG"][i]}\\t{counts["C_not_in_CpG"][i]}\\t{counts["G_not_in_CpG"][i]}\\n')
    logging.info(f'Saved: {output_file}')

print('Nucleotide counting functions defined.')

---
## Part 6: Parse CpG vs Non-CpG Mutations (Optional)

Separate SNPs into CpG-associated and non-CpG groups for detailed analysis.

In [ ]:
def filter_cpg_mutations(input_file, cpg_output, non_cpg_output):
    """
    Separate SNPs by CpG context.
    CpG mutations: C>T with G at +1, or G>A with C at -1
    """
    with open(input_file, 'r') as infile, \
         open(cpg_output, 'w') as cpg_out, \
         open(non_cpg_output, 'w') as non_cpg_out:
        for line in infile:
            columns = line.split()
            if len(columns) < 4:
                continue
            mutation = columns[1]
            flanking = columns[-1].strip()
            is_cpg = ((mutation == 'C>T' and len(flanking) > 2 and flanking[2] == 'G') or
                     (mutation == 'G>A' and len(flanking) > 0 and flanking[0] == 'C'))
            (cpg_out if is_cpg else non_cpg_out).write(line)

print('CpG filtering function defined.')